# 01 · Limpeza dos dados

Os dados vêm do [Brasileirao_Dataset](https://github.com/adaoduque/Brasileirao_Dataset), mantido pelo Adão Duque, que reúne as partidas da Série A desde 2003. Uso dois arquivos:

- `campeonato-brasileiro-full.csv`: uma linha por partida, de 2003 a 2024
- `campeonato-brasileiro-gols.csv`: uma linha por gol, com o minuto, de 2014 a 2024

Antes de analisar qualquer coisa, conferi o que precisava de ajuste. As funções de limpeza ficam em `src/dados.py` e este notebook mostra o porquê de cada uma.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.dados import (
    PASTA_PROCESSADOS,
    ler_gols_brutos,
    ler_partidas_brutas,
    limpar_gols,
    limpar_partidas,
)

In [2]:
partidas_raw = ler_partidas_brutas()
gols_raw = ler_gols_brutos()

print(f"Partidas: {partidas_raw.shape[0]} linhas, {partidas_raw.shape[1]} colunas")
print(f"Gols: {gols_raw.shape[0]} linhas, {gols_raw.shape[1]} colunas")

Partidas: 8785 linhas, 16 colunas
Gols: 9861 linhas, 6 colunas


In [3]:
partidas_raw.head()

,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado
0,1,1,29/03/2003,16:00,Guarani,Vasco,NaN,NaN,NaN,NaN,Guarani,Brinco de Ouro,4,2,SP,RJ
1,2,1,29/03/2003,16:00,Athletico-PR,Gremio,NaN,NaN,NaN,NaN,Athletico-PR,Arena da Baixada,2,0,PR,RS
2,3,1,30/03/2003,16:00,Flamengo,Coritiba,NaN,NaN,NaN,NaN,-,Maracanã,1,1,RJ,PR
3,4,1,30/03/2003,16:00,Goias,Paysandu,NaN,NaN,NaN,NaN,-,Serra Dourada,2,2,GO,PA
4,5,1,30/03/2003,16:00,Internacional,Ponte Preta,NaN,NaN,NaN,NaN,-,Beira Rio,1,1,RS,SP


## Partidas

### Valores ausentes

In [4]:
partidas_raw.isna().sum()

ID                       0
rodata                   0
data                     0
hora                     0
mandante                 0
visitante                0
formacao_mandante     4975
formacao_visitante    4975
tecnico_mandante      4610
tecnico_visitante     4610
vencedor                 0
arena                    0
mandante_Placar          0
visitante_Placar         0
mandante_Estado          0
visitante_Estado         0
dtype: int64

In [5]:
datas_raw = pd.to_datetime(partidas_raw["data"], format="%d/%m/%Y")
datas_raw[partidas_raw["tecnico_mandante"].notna()].min()

Timestamp('2014-04-19 00:00:00')

Só formação e técnico têm buracos, e os dois começam a ser preenchidos em 2014. Para as análises de agora isso não atrapalha, mas fica anotado para quando eu for usar essas colunas.

### O ano da partida nem sempre é a temporada

In [6]:
datas_raw.dt.year.value_counts().sort_index()

data
2003    552
2004    552
2005    462
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    268
2021    492
2022    380
2023    380
2024    380
Name: count, dtype: int64

2020 aparece com 268 jogos e 2021 com 492. Por causa da pandemia, o Brasileirão 2020 começou em agosto e só terminou em fevereiro de 2021, então parte dos jogos dele caiu no ano seguinte. A coluna `temporada` corrige isso: partidas de janeiro a abril de 2021 vão para a temporada 2020.

In [7]:
partidas = limpar_partidas(partidas_raw)
partidas["temporada"].value_counts().sort_index()

temporada
2003    552
2004    552
2005    462
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
Name: count, dtype: int64

Agora cada temporada fecha certinho. As exceções têm explicação: 2003 e 2004 tiveram 24 clubes e 2005 teve 22, por isso mais jogos. Em 2016 falta uma partida:

In [8]:
temporada_2016 = partidas[partidas["temporada"] == 2016]
clubes = temporada_2016["mandante"].unique()
confrontos = set(zip(temporada_2016["mandante"], temporada_2016["visitante"]))

[(casa, fora) for casa in clubes for fora in clubes if casa != fora and (casa, fora) not in confrontos]

[('Chapecoense', 'Atletico-MG')]

Chapecoense x Atlético-MG, da última rodada, não foi disputado depois do acidente aéreo com a delegação da Chapecoense. Nenhum dos dois times entrou em campo, então não há placar para registrar.

### Espaço invisível no nome das arenas

In [9]:
print("Jogos com arena == 'Maracanã':", (partidas_raw["arena"] == "Maracanã").sum())
print("Arenas que começam com \\xa0:", partidas_raw["arena"].str.startswith("\xa0").sum())
partidas_raw.loc[partidas_raw["arena"].str.contains("Maracan"), "arena"].iloc[0]

Jogos com arena == 'Maracanã': 0
Arenas que começam com \xa0: 5077


'\xa0Maracanã'

Filtrar por `"Maracanã"` não traz nenhum jogo, porque o nome vem com um espaço não separável (`\xa0`) na frente, coisa típica de dado raspado de página web. Ele não aparece na tela, mas quebra qualquer filtro ou junção por nome. Isso acontece em mais da metade das partidas, então troquei o caractere por espaço comum e tirei os espaços das pontas.

In [10]:
print("Jogos com arena == 'Maracanã' depois da limpeza:", (partidas["arena"] == "Maracanã").sum())

Jogos com arena == 'Maracanã' depois da limpeza: 669


### A coluna `vencedor` bate com o placar?

In [11]:
vencedor_pelo_placar = partidas["mandante"].where(
    partidas["resultado"] == "mandante",
    partidas["visitante"].where(partidas["resultado"] == "visitante", "-"),
)

colunas = ["data", "mandante", "visitante", "gols_mandante", "gols_visitante", "vencedor"]
partidas.loc[vencedor_pelo_placar != partidas["vencedor"], colunas]

,data,mandante,visitante,gols_mandante,gols_visitante,vencedor
1113,2005-04-24,Brasiliense,Vasco,0,1,-


Só um jogo diverge: Brasiliense x Vasco, em abril de 2005. Fui atrás e, segundo o [histórico do NetVasco](https://netvasco.com.br/historia/jogos/Jogos2005.html), a partida terminou 2x2 em campo, mas o STJD deu a vitória ao Vasco por 1x0. O placar da base segue a decisão do tribunal, e a coluna `vencedor` ficou com o empate de campo.

Como as análises usam pontos, calculei o resultado sempre a partir do placar oficial (colunas `resultado`, `pontos_mandante` e `pontos_visitante`).

### Partidas duplicadas

In [12]:
print("Confrontos repetidos na mesma temporada:", partidas.duplicated(subset=["temporada", "mandante", "visitante"]).sum())

Confrontos repetidos na mesma temporada: 0


Nenhum confronto se repete dentro da mesma temporada.

## Gols

### Minuto com acréscimo

In [13]:
print(gols_raw["minuto"].dtype)
gols_raw["minuto"].sample(10, random_state=1).tolist()

str


['55', '23', '67', '81', '13', '25', '90+2', '90+5', '49', '89']

O minuto vem como texto, e os gols nos acréscimos aparecem como `45+2` ou `90+3`. Separei em `minuto_base` e `acrescimo` e criei:

- `tempo`: 1º ou 2º tempo
- `faixa`: blocos de 15 minutos, com os acréscimos de cada tempo em faixas próprias (`45+` e `90+`)
- `minuto_ordem`: `45+2` vira 45,02, o que permite ordenar os gols na sequência real do jogo

In [14]:
gols = limpar_gols(gols_raw, partidas)
gols[["minuto", "minuto_base", "acrescimo", "tempo", "faixa", "minuto_ordem"]].sample(8, random_state=7)

,minuto,minuto_base,acrescimo,tempo,faixa,minuto_ordem
723,14,14,0,1,0-15,14.00
8797,90+1,90,1,2,90+,90.01
7727,90+3,90,3,2,90+,90.03
3488,78,78,0,2,76-90,78.00
378,33,33,0,1,31-45,33.00
56,11,11,0,1,0-15,11.00
8611,20,20,0,1,16-30,20.00
7522,33,33,0,1,31-45,33.00


In [15]:
gols["tipo_de_gol"].value_counts()

tipo_de_gol
Normal        8678
Penalty        935
Gol Contra     248
Name: count, dtype: int64

### Os gols batem com o placar?

Nos gols contra, a coluna `clube` traz o time que se beneficiou do gol, não o time do jogador. Para ter certeza disso (e de que não falta gol na lista), contei os gols de cada partida e comparei com o placar oficial.

In [16]:
contagem = gols.groupby("partida_id").agg(
    gols_mandante_lista=("gol_mandante", "sum"),
    total_lista=("gol_mandante", "size"),
)

conferencia = (
    partidas[partidas["temporada"] >= 2014]
    .set_index("id")
    .join(contagem)
    .fillna({"gols_mandante_lista": 0, "total_lista": 0})
)

total_bate = conferencia["total_lista"] == conferencia["gols_mandante"] + conferencia["gols_visitante"]
mandante_bate = conferencia["gols_mandante_lista"] == conferencia["gols_mandante"]

print(f"Partidas de 2014 a 2024: {len(conferencia)}")
print(f"Total de gols bate com o placar: {total_bate.mean():.0%}")
print(f"Gols do mandante batem com o placar: {mandante_bate.mean():.0%}")
print(f"Gols sem partida correspondente: {gols['temporada'].isna().sum()}")

Partidas de 2014 a 2024: 4179
Total de gols bate com o placar: 100%
Gols do mandante batem com o placar: 100%
Gols sem partida correspondente: 0


Bate em todas as partidas. A base de gols está completa para 2014 a 2024 e a atribuição dos gols contra está do jeito que eu esperava.

## Salvando os dados tratados

In [17]:
PASTA_PROCESSADOS.mkdir(exist_ok=True)
partidas.to_csv(PASTA_PROCESSADOS / "partidas.csv", index=False)
gols.to_csv(PASTA_PROCESSADOS / "gols.csv", index=False)

print(f"partidas.csv: {len(partidas)} linhas | gols.csv: {len(gols)} linhas")

partidas.csv: 8785 linhas | gols.csv: 9861 linhas


## Resumo

- Temporada 2020 reconstruída (parte dela foi jogada em 2021)
- Espaço invisível removido do nome das arenas
- Resultado e pontos calculados pelo placar oficial (um jogo de 2005 tinha `vencedor` divergente por decisão do STJD)
- Minuto dos gols separado em base e acréscimo, com faixas de 15 minutos
- Conferência: os 9.861 gols de 2014 a 2024 batem com o placar de todas as 4.179 partidas do período

Próximos notebooks: [02 · Fator casa](02_fator_casa.ipynb) e [03 · Gols por minuto](03_gols_por_minuto.ipynb).